# 05 - FISPACT activation studies

Solutions of the exercises. The setup cell below rebuilds the few objects they need, so this
notebook runs on its own.

In [ ]:
from pathlib import Path

import pandas as pd

from f4enix.core.constants import TIME_UNITS
from f4enix.core.irradiation import IrradiationScenario, Nuclide, Pulse, TCF_Computer
from f4enix.output.fispact_legacy_out import FispactOutput

DATA = Path("..") / "data"
FISP = DATA / "fispact"

pulsed = IrradiationScenario(
    [Pulse(2, 2.68e18, TIME_UNITS.YEAR)] + [Pulse(3290, 0), Pulse(400, 5.0e19)] * 17,
    name="two_years_then_pulses",
)
tcf = TCF_Computer()
nuclides = [Nuclide.from_formula("Co60"), Nuclide.from_formula("Mn56")]
fispact_out = FispactOutput(FISP / "testSS.out", cooling_times=["1e2 s", "1e4 s"])

---

# Exercises

## Exercise 1 - does the pulsing matter?

The `pulsed` scenario of section 5 delivers its fluence in two years of steady operation
followed by seventeen short pulses. A very common shortcut is to smear all of it into a single
steady irradiation of the same length. Find out for which daughters that shortcut is legitimate.

1. compute the total fluence and the total duration of `pulsed`, and build from them an
   equivalent scenario `steady` made of **one single pulse**, carrying the same fluence over the
   same duration;
2. give both scenarios the same cooling times: 1 hour, 1 day and 30 days;
3. compute the time correction factors of `Co60` and `Mn56` for both scenarios, with `norm=1.0`;
4. build a DataFrame `ratios`, indexed by the cooling labels and with one column per nuclide,
   holding the pulsed factor divided by the steady one. Which of the two nuclides notices the
   pulses, and why?

**Hints**
* `Pulse.time` is always stored in seconds, whatever unit was used to build the pulse, and
  `Pulse.intensity` is its flux - so the fluence of a pulse is the product of the two;
* `set_cooling_times([(1, TIME_UNITS.HOUR), ...])`, and the resulting `cooling_labels` are the
  index you want for `ratios`;
* `compute_correction_factors` wants the cooling times to be set already, and gives back an
  array with one row per cooling time and one column per nuclide;
* `tcf` and `nuclides` are the ones already built in section 6.

In [ ]:
# 1: what the pulsed scenario delivers, and over how long
fluence = sum(p.time * p.intensity for p in pulsed.pulses)
duration = sum(p.time for p in pulsed.pulses)

# 2: the same fluence smeared over the same time
steady = IrradiationScenario([Pulse(duration, fluence / duration)], name="steady_equivalent")

# 3: the two scenarios have to be looked at at the same moments
cooling = [(1, TIME_UNITS.HOUR), (1, TIME_UNITS.DAY), (30, TIME_UNITS.DAY)]
pulsed.set_cooling_times(cooling)
steady.set_cooling_times(cooling)

# 4: the correction factors of the two histories
factors_pulsed = tcf.compute_correction_factors(pulsed, nuclides, norm=1.0)
factors_steady = tcf.compute_correction_factors(steady, nuclides, norm=1.0)

# 5: how much the pulses change them
ratios = pd.DataFrame(
    factors_pulsed / factors_steady,
    index=pulsed.cooling_labels,
    columns=[nuclide.write_to_formula() for nuclide in nuclides],
)

print(f"fluence          : {fluence:.4e} n/cm2")
print(f"steady intensity : {fluence / duration:.4e} n/cm2/s")
ratios

In [ ]:
# --- self-check, do not modify ---
fluence_pulsed = sum(p.time * p.intensity for p in pulsed.pulses)
fluence_steady = sum(p.time * p.intensity for p in steady.pulses)

assert len(steady.pulses) == 1, (
    f"the steady scenario has {len(steady.pulses)} pulses, it should have exactly one"
)
assert abs(fluence_steady / fluence_pulsed - 1) < 1e-9, (
    f"the steady scenario carries {fluence_steady:.4e} n/cm2, the pulsed one {fluence_pulsed:.4e}"
)
assert pulsed.cooling_labels == steady.cooling_labels == ["1h", "1d", "30d"], (
    f"cooling labels are {pulsed.cooling_labels} and {steady.cooling_labels}, expected "
    "['1h', '1d', '30d'] on both"
)
assert ratios.shape == (3, 2), f"the ratio table is {ratios.shape}, expected (3, 2)"
assert abs(ratios["Co60"].to_numpy() - 1).max() < 0.05, (
    f"the Co60 ratios are {ratios['Co60'].to_numpy()}, they should all be close to 1"
)
assert ratios["Mn56"].min() > 1.5, (
    f"the Mn56 ratios are {ratios['Mn56'].to_numpy()}, the pulses should push them well above 1"
)
print("Exercise 1 correct")

Co60 lives 5.3 years: it integrates the whole history, so only the total fluence matters and
the ratio is 1. Mn56 lives 2.6 hours: at shutdown it is in equilibrium with the flux of the
last pulses, 5e19 instead of the 2.7e18 average, and the 3290 s dwell between them is only a
third of a half life - not enough to bring it back down. It ends up 2.3 times higher than the
smeared history predicts, at every cooling time.

The shortcut is safe for the long-lived inventory and wrong for everything that drives the
dose rate in the first hours after shutdown.

## Exercise 2 - dose or heat, same culprits?

The shutdown dose rate is a shielding problem, the decay heat is a cooling problem. Both come
out of the same inventory, but not necessarily out of the same nuclides. Check it on
`fispact_out`, 100 seconds after shutdown.

1. get the isotopes that make up 95% of the **dose** and those that make up 95% of the **decay
   heat**, both at the `"1e2 s"` cooling time, and call them `dose_drivers` and `heat_drivers`;
2. label the rows of the two tables with a `nuclide` column built from the element, the isotope
   and the state, then compute the two set differences `dose_only` and `heat_only` and print
   them together with the nuclides common to both lists;
3. build a table `comparison` with one row per nuclide of the **union** of the two lists,
   holding the `nuclide` label, its `isotope % dose` and its `isotope % heat`, plus a
   `heat/dose` column with the ratio of the two shares;
4. print the nuclide with the largest ratio. Why is it a cooling problem and not a shielding
   problem?

**Hints**
* `filter_by_cum_dose(95, "1e2 s")` and `filter_by_cum_heating(95, "1e2 s")`; the label must be
  written exactly as it was given to `FispactOutput`;
* the state is `""` for a ground state and `"m"` for a metastable, so
  `df["element"] + df["isotope"].astype(str) + df["state"]` is a good label;
* the shares of **all** the isotopes, not only of the drivers, are in `fispact_out.sddr` and
  `fispact_out.decay_heat`: filter them on the cooling time, then keep your union of nuclides;
* `pd.merge(dose_part, heat_part, on="nuclide")` puts the two share columns side by side.

In [ ]:
label = "1e2 s"


def add_nuclide(df: pd.DataFrame) -> pd.DataFrame:
    df["nuclide"] = df["element"] + df["isotope"].astype(str) + df["state"]
    return df


# 1: the two lists of drivers
dose_drivers = add_nuclide(fispact_out.filter_by_cum_dose(95, label).copy())
heat_drivers = add_nuclide(fispact_out.filter_by_cum_heating(95, label).copy())

# 2: who is in which list
dose_set = set(dose_drivers["nuclide"])
heat_set = set(heat_drivers["nuclide"])
dose_only = dose_set - heat_set
heat_only = heat_set - dose_set

print("both      :", sorted(dose_set & heat_set))
print("dose only :", sorted(dose_only))
print("heat only :", sorted(heat_only))

# 3: their two shares side by side, taken from the complete tables
union = dose_set | heat_set
dose_all = add_nuclide(fispact_out.sddr[fispact_out.sddr["cooling time"] == label].copy())
heat_all = add_nuclide(
    fispact_out.decay_heat[fispact_out.decay_heat["cooling time"] == label].copy()
)
comparison = pd.merge(
    dose_all[dose_all["nuclide"].isin(union)][["nuclide", "isotope % dose"]],
    heat_all[heat_all["nuclide"].isin(union)][["nuclide", "isotope % heat"]],
    on="nuclide",
    how="outer",
)
comparison["heat/dose"] = comparison["isotope % heat"] / comparison["isotope % dose"]
comparison = comparison.sort_values("heat/dose", ascending=False)

# 4: the one that heats more than it irradiates
worst = comparison.iloc[0]
print(
    f"\n{worst['nuclide']} is {worst['isotope % dose']:.2f}% of the dose "
    f"but {worst['isotope % heat']:.2f}% of the heat"
)
comparison

In [ ]:
# --- self-check, do not modify ---
assert len(dose_drivers) == 3, (
    f"the 95% dose table has {len(dose_drivers)} rows, expected 3"
)
assert len(heat_drivers) == 4, (
    f"the 95% heat table has {len(heat_drivers)} rows, expected 4"
)
assert set(dose_only) == set(), f"nothing drives the dose only, but you found {dose_only}"
assert set(heat_only) == {"Co57"}, f"the heat-only nuclide should be Co57, you found {heat_only}"
assert len(comparison) == 4, f"the comparison table has {len(comparison)} rows, expected 4"
for column in ("nuclide", "heat/dose"):
    assert column in comparison.columns, (
        f"no {column!r} column in comparison, its columns are {list(comparison.columns)}"
    )
worst = comparison.sort_values("heat/dose", ascending=False).iloc[0]
assert worst["nuclide"] == "Co57", f"the largest heat/dose ratio is {worst['nuclide']}, not Co57"
assert worst["heat/dose"] > 3, (
    f"Co57 carries {worst['heat/dose']:.2f} times its dose share as heat, expected close to 3.9"
)
print("Exercise 2 correct")

The two lists share Co60, Mn54 and Co58, and the heat list has one nuclide more: Co57.
It decays by electron capture and its strongest line sits at 122 keV, energy that is deposited
in the component but hardly reaches the worker - 4.9% of the decay heat against 1.3% of the
dose. Fe55, further down the table, is the extreme case of the same effect.

Cooling and shielding are therefore not driven by the same isotopes, and a nuclide list built
for one of the two problems cannot be reused for the other.